# Home Credit Agent Session

This notebook is a deterministic bootstrap for the DSBench notebook agent.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
TARGET = 'TARGET'
np.random.seed(SEED)


In [2]:
# The task says "already loaded Home Credit data".
# Let's assume app_train and app_test are already in memory.
# If not, we'll try to load them from the environment variable.
import os
from pathlib import Path

if 'app_train' not in globals():
    # Try to find the data directory relative to the notebook or in common locations
    # The previous error showed it was looking in /home/devang/Projects/PilotCrew/Data-Science-Bench/notebooks/data/home-credit-default-risk/
    # Let's try to find where application_train.csv actually is.
    
    found = False
    search_paths = [
        Path(os.environ.get('HOME_CREDIT_DATA_DIR', '')),
        Path('/kaggle/input/home-credit-default-risk'),
        Path('./data/home-credit-default-risk'),
        Path('../data/home-credit-default-risk'),
        Path('../../data/home-credit-default-risk')
    ]
    
    for p in search_paths:
        if p and p.exists() and (p / 'application_train.csv').exists():
            app_train = pd.read_csv(p / 'application_train.csv')
            app_test = pd.read_csv(p / 'application_test.csv')
            found = True
            print(f"Loaded data from {p}")
            break
            
    if not found:
        # Last resort: check if it's in the parent directory's data folder
        p = Path('data/home-credit-default-risk')
        if p.exists() and (p / 'application_train.csv').exists():
            app_train = pd.read_csv(p / 'application_train.csv')
            app_test = pd.read_csv(p / 'application_test.csv')
            found = True
            print(f"Loaded data from {p}")

if 'app_train' in globals():
    print(f'app_train shape: {app_train.shape}')
    print(f'TARGET positive rate: {app_train[TARGET].mean():.4f}')
else:
    print("Failed to load app_train. Please ensure data is available.")

Loaded data from ../data/home-credit-default-risk
app_train shape: (307511, 122)
TARGET positive rate: 0.0807


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# Preprocessing
# 1. Handle categorical variables
le = LabelEncoder()
le_count = 0

for col in app_train:
    if app_train[col].dtype == 'object':
        if len(list(app_train[col].unique())) <= 2:
            le.fit(app_train[col])
            app_train[col] = le.transform(app_train[col])
            app_test[col] = le.transform(app_test[col])
            le_count += 1

app_train = pd.get_dummies(app_train)
app_test = pd.get_dummies(app_test)

# Align the training and testing data, keep only columns present in both dataframes
train_labels = app_train['TARGET']
app_train, app_test = app_train.align(app_test, join='inner', axis=1)
app_train['TARGET'] = train_labels

# 2. Feature Selection (Deterministic: top N features by correlation with TARGET)
correlations = app_train.corr()['TARGET'].sort_values()
selected_features = correlations.abs().sort_values(ascending=False).iloc[1:11].index.tolist()

# 3. Prepare data for training
X = app_train[selected_features]
y = app_train['TARGET']

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Split data
X_train, X_val, y_train, y_val = train_test_split(X_imputed, y, test_size=0.2, random_state=SEED)

# 4. Train a deterministic binary classifier
trained_model = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
trained_model.fit(X_train, y_train)

# 5. Validation
y_pred_proba = trained_model.predict_proba(X_val)[:, 1]
y_pred = trained_model.predict(X_val)

auc = roc_auc_score(y_val, y_pred_proba)
acc = accuracy_score(y_val, y_pred)

training_metrics = {
    'auc': auc,
    'accuracy': acc,
    'feature_count': len(selected_features)
}

print(f"Selected Features: {selected_features}")
print(f"Feature Count: {training_metrics['feature_count']}")
print(f"Validation AUC: {training_metrics['auc']:.4f}")
print(f"Validation Accuracy: {training_metrics['accuracy']:.4f}")

/tmp/ipykernel_31861/353891886.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train['TARGET'] = train_labels


Selected Features: ['EXT_SOURCE_3', 'EXT_SOURCE_2', 'EXT_SOURCE_1', 'DAYS_BIRTH', 'REGION_RATING_CLIENT_W_CITY', 'REGION_RATING_CLIENT', 'NAME_INCOME_TYPE_Working', 'NAME_EDUCATION_TYPE_Higher education', 'DAYS_LAST_PHONE_CHANGE', 'CODE_GENDER_M']
Feature Count: 10
Validation AUC: 0.7021
Validation Accuracy: 0.9186


In [4]:
# Check if model and data are in memory
model_exists = 'trained_model' in globals()
features_exists = 'selected_features' in globals()
test_exists = 'app_test' in globals()

print(f"Model in memory: {model_exists}")
print(f"Selected features in memory: {features_exists}")
print(f"app_test in memory: {test_exists}")

if model_exists and features_exists and test_exists:
    # Prepare the first 10 rows of app_test
    X_test_sample = app_test[selected_features].head(10)
    
    # Reuse the imputer from the previous cell if it exists, 
    # otherwise we might need to re-instantiate or check if it was saved.
    # Looking at cell 3, 'imputer' was defined.
    if 'imputer' in globals():
        X_test_sample_imputed = imputer.transform(X_test_sample)
    else:
        # Fallback if imputer not in globals (though it should be)
        from sklearn.impute import SimpleImputer
        temp_imputer = SimpleImputer(strategy='median')
        X_test_sample_imputed = temp_imputer.fit_transform(X_test_sample)
        
    # Score the first 10 rows
    probs = trained_model.predict_proba(X_test_sample_imputed)[:, 1]
    
    # Get top 5 features by importance
    importances = pd.Series(trained_model.feature_importances_, index=selected_features)
    top_5_features = importances.sort_values(ascending=False).head(5)
    
    print("\nScores (probabilities) for the first 10 rows of app_test:")
    print(probs)
    
    print("\nTop 5 Features by Importance:")
    print(top_5_features)
else:
    print("Required artifacts not found in memory.")

Model in memory: True
Selected features in memory: True
app_test in memory: True

Scores (probabilities) for the first 10 rows of app_test:
[0.01 0.08 0.01 0.09 0.13 0.02 0.02 0.11 0.01 0.08]

Top 5 Features by Importance:
EXT_SOURCE_2              0.245427
DAYS_BIRTH                0.223586
DAYS_LAST_PHONE_CHANGE    0.188345
EXT_SOURCE_3              0.172980
EXT_SOURCE_1              0.109287
dtype: float64
